# GraphRAG — demo

Composição real de `rag/` (Chroma + embeddings) e `knowledge_graph/` (extração de entidades/relações via LLM + travessia de grafo). Ver `rag/GRAPH_RAG.md` para a arquitetura completa.

**Requer um provedor de LLM configurado** (Ollama local por padrão — `ollama serve` + `ollama pull mistral`) para a extração de entidades/relações. Os testes unitários (`rag/tests/test_graph_rag.py`) usam fakes e não precisam disso.

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('.'))
from memory.vector_memory import VectorMemory
from rag.graph_rag import GraphRAGPipeline

pipeline = GraphRAGPipeline(vector_store=VectorMemory(persist_dir='vector_db/graph_rag_demo'))


## Ingestão

Cada documento é indexado nos dois lados ao mesmo tempo: chunk+embedding (Chroma) e entidades+relações (grafo).

In [ ]:
docs = [
    ('Marie Curie trabalhou com Pierre Curie na Sorbonne e descobriu o polônio e o rádio.', 'bio_curie.txt'),
    ('O Argus é uma plataforma de Customer Intelligence com Lakehouse, MDM e agentes de IA.', 'argus_overview.txt'),
]
for text, source in docs:
    pipeline.ingest(text, source=source)

print(pipeline.stats())


## Consulta relacional

O ponto onde GraphRAG ganha de RAG puro: a pergunta é sobre uma **relação** (onde trabalhou), não sobre similaridade textual.

In [ ]:
context = pipeline.retrieve('Onde Marie Curie trabalhou?')
print('--- trechos vetoriais ---')
print(context.vector_chunks)
print('
--- contexto do grafo ---')
print(context.graph_context)
print('
--- contexto final (prompt) ---')
print(context.render())


## Consulta factual/semântica

Para comparação: uma pergunta que a busca vetorial sozinha já resolve bem.

In [ ]:
context2 = pipeline.retrieve('O que é o Argus?')
print(context2.render())


## Nota

Em produção, `context.render()` entra no prompt do LLM junto com a pergunta do usuário — o mesmo padrão usado pelo restante do `agents/` deste projeto.